# Disclaimer

This is not a real tutorial but more a generic steps guide about how to process MS data (specially LCMS). 

The steps below are something you should think about, yet not to be strickly followed. every dataset can be slightly different. 

Always check your data and see what you need in each step


<br>

also, the steps are more for non-targeted analysis and thus there will be a lot of uncertainties

------------------------------

Last update: 2026-09-17

# 0. the data collection step

this is more of a reminder that how you collect your data will impact how the processing (especially on filtering)

for example: 
- do you have multiple blank injections, and are they injected in sequence or between samples?
- do you have internal standard? just one or multiple? does it match with the compound class in the sample? 
- do you have a QC sample like a pool sample of all sample of the same type/all sample you have/a representative standard mix? 
- do you have time to run multiple injection of your samples or only one injection per sample? 


all of these will cost extra time or preperation so you may not have all of them. but it's always good to have at least one (for example, a few blanks)

# 1. Once you get the raw data

first thing first, load your data, at least blank and a few sample files to make sure it looks right to you

i.e., no weird peak shapes (or you should see the expectd blurb of signal for BBOA sampes in the middle of the run)

blank looks ok overall etc 


if you only need a few things on the software, you may use it as it is. and stop here. 




# 2. get a peak table 


<Br>

usually this go from _feature detection > alignment between samples_ 


a lot of time we call a peak "feature", as it can be combination of many things but appear to be a peak on chromatogram. 

<br>
feature detection is basically "constructing a peak" from the raw data, similar to centroiding for mass signal (though not exactly). from a peak you see on chromatogram, usually we need to link a series of mass signals. instrument mass resolution do matter in this case, as this affects how signals are binned together, which then determine the peak shape, width etc. at the end, you get a peak with its RT, m/z, intensity/area. 


this happens within a sample data from the begining to the end. Since we have multiple samples, we need to align them together to get to the final table .


usually there are lots of software to do it. pick whichever you like. 

I just happened to use MZMine (checkout the 2_mzmine_tutorial if you need), but there are others like *msdial*, *openms*, or  Mass spec vendors have their own softwares 

or write the code if you want (such as pyopenms package)



<br>

a lot of time software will do "gap filling". this means that a peak might not be detected in some samples (while it's detected in others) due to various filter/constrains.

the software will go back to the raw data and see if the signal is actually there but just gets filtered/ignored. 

in mzmine this will make the peak noted as "ESTIMATED" instead of "DETECTED" (which means it pass all the filters and succesfully constructed as a peak in the software)

------------------------

now, the steps below are just items you can do but doesnt have to follow the order. 

for example you may run PCA on your data before and after filtering or you might want to use formula to filter your data. these are all valid and reasonable steps which you can use. there's no right or wrong as you know why you're doing them. 


--------------------------

# 3. filtering your data


so, you get a table with lots of information. you might want to filter out peaks that are not important or you dont want to see them, such as noise peak, blank peaks etc. 

NOTE: THIS STEPS TAKES A BIT OF THINKING AS THE THRESHOLDS ARE ARBITARY AND THERE'S NO ONE NUMBER FITS ALL DATA.


<Br>

### 3.1 - Occurence or group filtering

a general rule is that, if you have multiple injection of the same sample, a real signal should be relatively consistent and appear in most of them. 

this is similar for a group of samples (lets say you chop the wood into 3  and collected 3 burns). they should be somewhat similar. 

so we can make a rule that `the peak has to appear in x% of the sample in a group` which the group can be from samples or experimental replicates 

for example: 
- a peak is only retained if it appear in at leat 50% of at least one sample group
- a peak is filtered if it appeared in at least 75% of  at least one of sample replicate group and the intensity difference shouldnt be above X%


<Br><Br>

###  3.2- Blank filter

Running blank is pretty much an always. lets assume you have method blank and run it three times. 

blank filtering has a few different steps, commonly: defining blank peak > filtering or blank subtraction


##### define blank peaks

same as the previous occurence/group filtering, a real blank peak subject to the same idea:

`a real blank peak should be at least x% of all blanks and intensity varaince should be within Y`


which you need to define X and Y. X is much easier and usually something like 75% or 80%. 

the variance is more tricky and depend on your data. so manually check data at the first step is important

or use an arbitary number like 30% (LCMS usually have a 10% variation on good contorlled Standard run, so samples would be higher)



<br><br>
from previous step you define blank peaks, then you can filter peaks out based on intensity between the samples and blank. 

an arbitary number is 3 time higher (this has nothing to do with LOD but just borrow the number here and it's not too aggressive or conservative. of course you may use other numbers like 5 times or 2 times). 



<br>
blank subtraction is a bit more tricky. if your blank signals has pretty good agreement (like +/- 10%) you may just use average. 

but what if the blank signal varies a bit (but not too much that failed the blank peak check before), you might need to think about 
1) does average make sense
2) do you want to be more conservative and use the maximum or median in blank 
3) or, if the blank injections were between the sample injections (i.e., not  multiple times in sequence), you may think about using the blank before and after. this might works better if you have a large batch of data. 






# 4. quick check on data 

a few things you can check the data without much effort:


- plot peaks in RT vs m/z value with intensity on color scale. 
 
- or in software usually you have the option to view individual data or the aligned peak table, oftenly should show you the peak shapes and sometimes box plot etc. 

- PCA is a generic tool that i like to use before and after filtering. 



<Br>

Note that we dont inlcude normalization in the workflow but it can be important in lots of different situations. 

if needed, now is a good time to do normalization (since you have less peaks)

unless your data required to be normalizated as the first step 

# 5.formula prediction 

-----

see ***python > Code > 03_formula_matching*** if you want to do it on your own

-----

<br>

most of the software has this function already, with a defined limit of maximum number of each element (and which elements to use). note that not all software validate the predicted formula the same. 


for example, MZMine has the formula prediction function as well as for ion identity group. it will return the formula, the mass and error, isotope score and the combined score. 

isotope check is always a good thing to have (especially for Sulfur) and thus isotope score can be useful. however, the isotope signal will be something like 15% of the peak itself, so this is subjected to signal filtering. 

there are other genierc validating function to use :

- `DBE`: set range (like -1 or 0 up to 20 or 40 ) and has to be integer 

- `H/C or O/C ratio`: this is more toward understanding of the possible compounds in the sample. you need prior knowledge on this 

- `nitrogen rule`: this is interesting thing. nitrogen rule usually works well for small compounds but there are conditions that it will fail: 



- isotope score: this is provided from software from 0 to 1 in mzmine, basically calculating the simlarity between observed and theoratical pattern. 


<br>

the interesting thing for MZMine ion identity group is that you may find formula assigned for the group, but each individual peak may have different formula from the formula prediction module (as it only work on the mz value). you need to decicde which to use based on the error, isotope, and combined score. it's just an extra step to check which of course stil may have error and uncertainty. 

see ***Tutorial>2_mzmine_tutorial***
